# KernelBench_X — Quick-Start Notebook

Run three types of submissions through the full **call → exe → perf** pipeline and read the results.

> **Evaluation stages**
> | Stage | What it checks |
> |-------|----------------|
> | `call` | Syntax, quantization heuristic, kernel entry point |
> | `exe`  | Numerical accuracy vs. reference (`data/kernelbenchx/`) |
> | `perf` | Runtime vs. `metrics/golden_results/5090/` (→ `speedup`) |


## 0. Setup

Install dependencies and set `PYTHONPATH` so the evaluation scripts can locate `data/`.
Run this cell once before anything else.


In [ ]:
import os, sys
from pathlib import Path

# Make sure we are at the repo root (`.../KernelBenchX`).
# If running this cell prints a wrong path (e.g. `/root`), edit the next line:
# - change `".."` to the correct repo root path, or
# - comment it out and `cd` to the repo root manually.
os.chdir("..")

REPO = Path.cwd()
print("repo:", REPO)

# Install deps
!{sys.executable} -m pip install -r requirements.txt -q

# PYTHONPATH so eval scripts find data/kernelbenchx/
os.environ["PYTHONPATH"] = str(REPO / "data") + ":" + os.environ.get("PYTHONPATH", "")
print("PYTHONPATH set.")


## 1. JSONL Input

One JSON object per line with a `predict` field (implementation) and a `file` field
(dataset task path, e.g. `Activation/gelu.py`).

**Source:** `examples/sources/predictions.gelu.jsonl`  
**Output:** `examples/out/jsonl/`


In [ ]:
import os
from pathlib import Path

REPO = Path.cwd()
os.environ['PYTHONPATH'] = str(REPO / 'data') + ':' + os.environ.get('PYTHONPATH', '')

out = REPO / 'examples' / 'out' / 'jsonl'
!rm -rf "{out}"
!bash scripts/run_eval.sh examples/sources/predictions.gelu.jsonl "{out}" 0


## 2. Single `.py` Input

The file basename must match the task name (e.g. `matmul_w8a8.py`).

**Source:** `examples/sources/matmul_w8a8.py`  
**Output:** `examples/out/py/`


In [ ]:
import os
from pathlib import Path

REPO = Path.cwd()
os.environ['PYTHONPATH'] = str(REPO / 'data') + ':' + os.environ.get('PYTHONPATH', '')

out = REPO / 'examples' / 'out' / 'py'
!rm -rf "{out}"
!bash scripts/run_eval_from_files.sh examples/sources/matmul_w8a8.py "{out}" 0


## 3. Directory Input (mixed `.py` + `.jsonl`)

Directory mode walks a folder and evaluates every `.py` and `.jsonl` it finds.

```
examples/sources/
├── matmul_w8a8.py                    # Quantization — W8A8 group-wise int8 matmul
└── predictions.gelu.jsonl            # Activation  — GeLU
```

**Output:** `examples/out/dir/`


In [ ]:
import os
from pathlib import Path

REPO = Path.cwd()
os.environ["PYTHONPATH"] = str(REPO / "data") + ":" + os.environ.get("PYTHONPATH", "")

out = REPO / "examples" / "out" / "dir"
!rm -rf "{out}"
!bash scripts/run_eval_from_files.sh examples/sources "{out}" 0


## 4. Reading Results

Every run writes:

| File | Contents |
|------|----------|
| `metrics.json` | Per-task full results (call / exe / perf, errors, perf details) |
| `summary.json` | Aggregate stats (pass rates, average speedup) |
| `intermediate/` | Per-stage JSONL, generated perf scripts, perf logs |

**Key fields in `metrics.json`:**

- `call_ok` — passed syntax + quantization heuristic + kernel entry check
- `exe_ok` — output matches reference within tolerance
- `perf_ok` — runtime benchmark succeeded
- `speedup` — ratio of golden runtime to your implementation's runtime (> 1 = faster)
- `error` — human-readable failure reason